# Project Report: Polish Road-Sign Recogniser

## 1. Motivation
The goal of this project is to recognize Polish road signs from images. Because Polish and German road signs share high visual similarity, the German Traffic Sign Recognition Benchmark (GTSRB) dataset is used as a proxy for training and evaluation. The project is constrained to be trained end-to-end on a single CPU/modest GPU in under an hour, using a small dataset and an efficient model architecture (~200K parameters).

## 2. Related Work & Approach
We draw upon Chapters 22–25 and 27 (data augmentation) to structure our approach. We implement a Spatial Transformer Network (STN) on top of a Convolutional Neural Network (CNN). The STN learns an affine transformation to normalize the input image, allowing the CNN to be invariant to spatial transformations like scaling, cropping, and rotation.

## 3. Data
The GTSRB dataset contains ~50,000 training images across 43 classes. We apply data augmentation techniques such as random rotations, skews, and color jitter to improve robustness. Images are resized to 48x48 resolution to capture finer details.

## 4. Model Architecture
The model consists of:
1. **STN:** A localization network followed by a regressor that predicts a 2x3 affine transformation matrix.
2. **CNN:** A 4-layer CNN with 3 max-pooling layers. This architecture achieves a parameter count of ~240,000, fitting our constraints while maintaining high accuracy.

## 5. Training Details & Results
The model is trained using the **AdamW** optimizer with decoupled weight decay. We also calculate class weights to handle the inherent imbalance in the GTSRB dataset. Training completes efficiently within the one-hour constraint.

In [ ]:
# Let's load the trained model and check its parameters
import torch
from model import STN_CNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = STN_CNN(num_classes=43).to(device)
model.load_state_dict(torch.load('model.pt', map_location=device))

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params}")


## 6. Error Analysis
During testing, we discovered that a 32x32 resolution struggled to recognize stop signs in live feeds. We increased the resolution to 48x48, which improved accuracy significantly. However, there are still failure modes (~40% on difficult live captures) due to lighting variations, motion blur, and the slight domain gap between German and Polish road signs.

## 7. Wow Angle: Sliding-Window Detection
To demonstrate the model's capabilities beyond simple classification, we implemented a sliding-window detection mechanism. It sweeps across a larger image at multiple scales, extracts patches, and feeds them to the model. We then apply Non-Maximum Suppression (NMS) to collapse overlapping bounding boxes.

In [ ]:
# Example of running sliding-window detection (Requires an image URL)
# !python demo.py --url https://example.com/road_sign.jpg


## 8. Lessons Learned
- **Resolution vs. Parameters:** A slight increase in input resolution (32 to 48) provides a necessary boost in feature extraction, but requires careful tuning of the pooling layers to keep the parameter count within budget.
- **Class Imbalance:** GTSRB is heavily imbalanced; using class weights in the Cross-Entropy Loss stabilizes training.
- **Real-world Testing:** Lab metrics can be deceiving. Live testing on actual Polish road signs highlighted the need for better resolution and highlighted the domain gap.